**Install Dependencies**

In [4]:
# Install the required packages
!pip install -qU langchain langchain-openai python-dotenv pydantic

**Environment Configuration & LangSmith Tracing**

In [ ]:
import os
from google.colab import userdata

print("Fetching API Keys from Colab Secrets...")
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGCHAIN_API_KEY')

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "GenAI_Resume_Screener"
print("LangSmith tracing enabled and keys loaded securely.")

**Prompt Engineering**

In [ ]:
from langchain_core.prompts import PromptTemplate

resume_evaluation_template = """
You are an expert technical recruiter evaluating a candidate's resume against a specific job description.

Job Description:
{job_description}

Candidate Resume:
{resume_text}

Analyze the resume and provide the output strictly as a JSON object matching the requested structure.

CRITICAL RULES:
1. Do NOT assume or hallucinate skills, tools, or experience not explicitly present in the resume.
2. Be highly objective.
3. The fit_score must be an integer between 0 and 100.

Provide your response in the following JSON format:
{{
    "extracted_data": {{
        "skills": ["skill1", "skill2"],
        "tools": ["tool1", "tool2"],
        "experience_summary": "Brief summary of experience"
    }},
    "matching_analysis": {{
        "matched_requirements": ["req1", "req2"],
        "missing_requirements": ["req3"]
    }},
    "fit_score": 85,
    "explanation": "Detailed reasoning for why this score was assigned based on the matches and missing elements."
}}
"""

eval_prompt = PromptTemplate(
    input_variables=["job_description", "resume_text"],
    template=resume_evaluation_template
)
print("Prompt Template successfully created.")

**LangChain Expression Language (LCEL) Pipeline**

In [ ]:
!pip install -qU langchain-groq

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

output_parser = JsonOutputParser()

screening_chain = eval_prompt | llm | output_parser

print("LCEL Screening Chain successfully built with updated model.")

**Test Data**

In [13]:
# 1. Define the Job Description
JOB_DESCRIPTION = """
Role: Data Scientist / AI Engineer
Requirements:
- Strong proficiency in Python (Core and Advanced modules).
- Experience with Natural Language Processing (NLP) and models like BERT or Transformers.
- Familiarity with LangChain and building LLM pipelines.
- Data manipulation skills using Pandas and NumPy.
- Ability to build REST APIs using FastAPI.
"""

# 2. Define the Candidates (Matching the 3 types required by the assignment)
RESUME_STRONG = """
Name: Akash
Experience: Data Science Intern
Skills: Python, C++, NumPy, Pandas, FastAPI, LangChain, LLMs.
Projects:
- Built an AI Resume Screening System using LangChain and LangSmith.
- Fine-tuned BERT and Transformer models for NLP sentiment analysis tasks.
- Created a portfolio website and deployed REST APIs using FastAPI.
"""

RESUME_AVERAGE = """
Name: Rahul
Experience: Junior Analyst
Skills: Python, SQL, Excel, basic Pandas.
Projects:
- Wrote basic Python scripts for data cleaning.
- Maintained SQL databases and generated Excel reports for management.
- Interested in learning machine learning.
"""

RESUME_WEAK = """
Name: Vikram
Experience: Graphic Designer & Social Media Manager
Skills: Adobe Photoshop, Illustrator, Pinterest Marketing, HTML, CSS.
Projects:
- Designed high-conversion pins for home decor products.
- Reached 10,000 followers on Instagram through content creation.
- Enhanced vintage photographs using lighting techniques.
"""

candidates = {
    "Strong Candidate": RESUME_STRONG,
    "Average Candidate": RESUME_AVERAGE,
    "Weak Candidate": RESUME_WEAK
}
print("Test Data loaded successfully.")

Test Data loaded successfully.


**Execute and Evaluate**

In [14]:
import json

print("Starting AI Resume Screening Evaluation...\n")

for candidate_type, resume in candidates.items():
    print(f"{'='*60}")
    print(f"Evaluating: {candidate_type}")
    print(f"{'='*60}")

    try:
        # Execute the pipeline
        result = screening_chain.invoke({
            "job_description": JOB_DESCRIPTION,
            "resume_text": resume
        })

        # Print the structured JSON output nicely
        print(json.dumps(result, indent=4))
        print("\n")

    except Exception as e:
        print(f"An error occurred during pipeline execution for {candidate_type}: {e}\n")

print("Evaluation Complete! Check your LangSmith Dashboard for the trace logs.")

Starting AI Resume Screening Evaluation...

Evaluating: Strong Candidate
{
    "extracted_data": {
        "skills": [
            "Python",
            "C++",
            "NumPy",
            "Pandas",
            "FastAPI",
            "LangChain",
            "LLMs"
        ],
        "tools": [
            "LangChain",
            "LangSmith",
            "BERT",
            "Transformers",
            "FastAPI"
        ],
        "experience_summary": "Data Science Intern with experience in building AI systems, fine-tuning NLP models, and deploying REST APIs."
    },
    "matching_analysis": {
        "matched_requirements": [
            "Strong proficiency in Python (Core and Advanced modules)",
            "Familiarity with LangChain and building LLM pipelines",
            "Ability to build REST APIs using FastAPI"
        ],
        "missing_requirements": [
            "Experience with Natural Language Processing (NLP) and models like BERT or Transformers",
            "Data